# Build the reusable Kaggle T4 runtime
Select **GPU T4 x2** and **Internet on**; attach exactly one private `vembody-kaggle.zip` source dataset, then run all three code cells.
This compiles/exports the pinned llama.cpp CUDA server only: no model weights, inference, training, or public endpoint. Compilation uses CPU and may take an hour or more while GPU session quota is consumed.
If you already have Josh's trusted `vembody-runtime-t4.zip`, skip this notebook and attach it to `kaggle-ui-mate-q6.ipynb` instead. See `notebooks/README.md` for the full handoff.

In [ ]:
import hashlib
import json
import shutil
import subprocess
import sys
from pathlib import Path
from zipfile import ZipFile

archives = list(Path("/kaggle/input").rglob("vembody-kaggle.zip"))
manifests = list(Path("/kaggle/input").rglob("bundle-manifest.json"))
if len(archives) == 1 and not manifests:
    archive = archives[0]
    snapshot = hashlib.sha256(archive.read_bytes()).hexdigest()
    destination = Path("/kaggle/temp") / ("vembody-bundle-" + snapshot[:12])
    destination.mkdir(parents=True, exist_ok=True)
    with ZipFile(archive) as bundle:
        if any(
            not (destination / item.filename).resolve().is_relative_to(destination)
            for item in bundle.infolist()
        ):
            raise RuntimeError("Unsafe archive member")
        bundle.extractall(destination)
    REPO = destination / "vembody"
elif len(manifests) == 1 and not archives:
    snapshot = hashlib.sha256(manifests[0].read_bytes()).hexdigest()
    REPO = Path("/kaggle/temp") / ("vembody-bundle-" + snapshot[:12]) / "vembody"
    if not REPO.exists():
        shutil.copytree(manifests[0].parent, REPO)
else:
    raise RuntimeError(
        "Attach exactly one Vembody bundle; both ZIP and Kaggle-unpacked inputs are supported"
    )
manifest = json.loads((REPO / "bundle-manifest.json").read_text())
for name, expected in manifest["files"].items():
    if not (REPO / name).resolve().is_relative_to(REPO):
        raise RuntimeError("Unsafe manifest path")
    if hashlib.sha256((REPO / name).read_bytes()).hexdigest() != expected:
        raise RuntimeError("Bundle checksum mismatch: " + name)
print("Source snapshot:", snapshot)

In [ ]:
sys.path.insert(0, str(REPO / "scripts"))
sys.modules.pop("kaggle_environment", None)
from kaggle_environment import create_environment  # noqa: E402 - load this verified bundle first.

ENV = Path("/kaggle/temp") / ("vembody-env-" + snapshot[:12])
PYTHON = create_environment(REPO, ENV)
WORK = Path("/kaggle/temp/ui-mate-q6-runtime")
RUNNER = REPO / "scripts/kaggle_run.py"
CACHE = Path("/kaggle/working/vembody-runtime-t4.zip")
print("Isolated Python:", PYTHON)

In [ ]:
subprocess.run(
    [
        str(PYTHON),
        "-I",
        str(RUNNER),
        "build-runtime",
        "--repo",
        str(REPO),
        "--work",
        str(WORK),
        "--cache-output",
        str(CACHE),
    ],
    check=True,
)
if not CACHE.is_file():
    raise RuntimeError("Expected runtime ZIP was not exported")
print("Runtime ZIP:", CACHE, "bytes:", CACHE.stat().st_size)
print("Download from Output > /kaggle/working > file menu > Download.")
print("Upload the ZIP as a PRIVATE input for the experiment notebook; share only with your team.")
print("After downloading, STOP this session to stop spending GPU quota.")